[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sk-2103/AI4Sustainability-Labs/blob/main/labs/lab06_gfm_flood_mapping/lab06_evaluation.ipynb)  
**AI for Sustainability** · University of Wisconsin–Madison · Labs by Saurabh Kaushik · Course instructor: Dr. Beth Tellman  
*Make a copy (File → Save a copy in Drive) before you start. Part of the [AI for Sustainability lab series](https://github.com/Sk-2103/AI4Sustainability-Labs).*

<div style="background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);padding:36px 28px;border-radius:14px;margin-bottom:8px">
  <h1 style="color:#00e5ff;font-family:'Courier New',monospace;font-size:2em;margin:0 0 8px 0">📝 Lab 06 — Assessment</h1>
  <h2 style="color:#e0f7fa;font-weight:300;margin:0 0 16px 0;font-size:1.2em">Geospatial Foundation Models · Flood Mapping with TerraMind</h2>
  <p style="color:#b2ebf2;margin:0 0 6px 0;font-size:.9em">AI for Sustainability · University of Wisconsin–Madison</p>
  <div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:14px">
    <span style="background:rgba(0,229,255,.15);color:#00e5ff;padding:3px 14px;border-radius:20px;font-size:.85em;border:1px solid #00e5ff">Total: 25 marks</span>
    <span style="background:rgba(255,183,77,.15);color:#ffb74d;padding:3px 14px;border-radius:20px;font-size:.85em;border:1px solid #ffb74d">Theory: 12 marks</span>
    <span style="background:rgba(102,187,106,.15);color:#66bb6a;padding:3px 14px;border-radius:20px;font-size:.85em;border:1px solid #66bb6a">Practical: 13 marks</span>
  </div>
</div>

---

**Instructions**
- Answer **all** questions.
- **Theory:** write answers in the markdown cells directly below each question.
- **Practical:** fill in every blank marked `### YOUR CODE HERE ###`. Do **not** change any other lines. Each question references the relevant Lab 06 section so you can look it up.
- Run cells **top to bottom** — each section depends on the ones above.
- Submit this `.ipynb` with all cells executed.

| Section | Questions | Marks |
|---------|-----------|-------|
| Theory  | Q1 – Q6   | 12    |
| Practical | P1 – P3 | 13    |
| **Total** |         | **25** |


---
## ⚙️ Step 0 — Install & Setup
*Same as Lab 06, Step 0. Run once, then **Runtime → Restart session**, then continue from Step 1.*
> 💡 **Tip:** If the runtime already has terratorch from the lab session, skip directly to Step 1.


In [ ]:
# Run once → restart session → skip this cell next time
import subprocess, sys

def run(cmd):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stderr[-500:])
    return result.returncode

run("apt-get update -qq")
run("apt-get install -qq -y libgdal-dev gdal-bin")
run("pip uninstall -y numpy scipy")
run("pip install -q --no-cache-dir 'numpy==1.26.4' 'scipy<1.14'")
run("pip install -q --no-cache-dir GDAL==$(gdal-config --version) rasterio")
run("pip install -q --no-cache-dir albumentations gdown tensorboard rich")
run("pip install -q --no-cache-dir 'terratorch==1.2.6' 'setuptools<81'")
print("\nDone — now RESTART the runtime.")


---
## ✅ Step 1 — Verify Environment
*Run after restarting. Identical to Lab 06 Step 0 verification cell.*


In [ ]:
import sys, warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import torch
import rasterio
import terratorch
import lightning.pytorch as pl
from importlib.metadata import version

print(f"Python     : {sys.version.split()[0]}")
print(f"NumPy      : {np.__version__}")
print(f"PyTorch    : {torch.__version__}")
print(f"TerraTorch : {version('terratorch')}")
print(f"Lightning  : {pl.__version__}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {DEVICE}")
if DEVICE.type == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {torch.cuda.get_device_name(0)}  ({vram:.1f} GB)")
    BACKBONE = "terramind_v1_small" if vram >= 10 else "terramind_v1_tiny"
else:
    BACKBONE = "terramind_v1_tiny"
    print("⚠️ No GPU — enable via Runtime → Change runtime type → T4 GPU")
print(f"Backbone   : {BACKBONE}")
print("\n✅ Ready")

---
## 📥 Step 2 — Download Sen1Floods11
*Same download as Lab 06 Step 1. Skip if data is already present from the lab session.*


In [ ]:
import os, gdown

DATASET_TAR = "sen1floods11_v1.1.tar.gz"
DATASET_DIR = "sen1floods11_v1.1"
GDRIVE_ID   = "1lRw3X7oFNq_WyzBO6uyUJijyTuYm23VS"

if not os.path.isdir(DATASET_DIR):
    if not os.path.isfile(DATASET_TAR):
        print("Downloading Sen1Floods11 (~1.4 GB)...")
        gdown.download(f"https://drive.google.com/uc?id={GDRIVE_ID}", DATASET_TAR, quiet=False)
    print("Extracting...")
    os.system(f"tar -xzf {DATASET_TAR}")
else:
    print(f"✅ Dataset already present at ./{DATASET_DIR}/")

S2_DIR  = f"{DATASET_DIR}/data/S2L1CHand"
S1_DIR  = f"{DATASET_DIR}/data/S1GRDHand"
LBL_DIR = f"{DATASET_DIR}/data/LabelHand"

with open(f"{DATASET_DIR}/splits/flood_train_data.txt") as f:
    TRAIN_NAMES = [l.strip() for l in f if l.strip()]
with open(f"{DATASET_DIR}/splits/flood_valid_data.txt") as f:
    VAL_NAMES   = [l.strip() for l in f if l.strip()]
with open(f"{DATASET_DIR}/splits/flood_test_data.txt") as f:
    TEST_NAMES  = [l.strip() for l in f if l.strip()]

print(f"Train: {len(TRAIN_NAMES)} | Val: {len(VAL_NAMES)} | Test: {len(TEST_NAMES)}")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

def pct_stretch(arr, lo=2, hi=98):
    a, b = np.percentile(arr, [lo, hi])
    return np.clip((arr - a) / (b - a + 1e-8), 0, 1)

def load_chip(name):
    with rasterio.open(f"{S2_DIR}/{name}_S2Hand.tif")  as f: s2  = f.read().astype(np.float32)
    with rasterio.open(f"{S1_DIR}/{name}_S1Hand.tif")  as f: s1  = f.read().astype(np.float32)
    with rasterio.open(f"{LBL_DIR}/{name}_LabelHand.tif") as f: lbl = f.read(1).astype(np.int16)
    return s2, s1, lbl

FLOOD_CMAP = ListedColormap(["#2d6a4f", "#0077b6"])

MEANS = {
    "S2L1C": [2357.089, 2137.385, 2018.788, 2082.986, 2295.651,
              2854.537, 3122.849, 3040.560, 3306.481, 1473.847,
              506.070,  2472.825, 1838.929],
    "S1GRD": [-12.599, -20.293],
}
STDS = {
    "S2L1C": [1624.683, 1675.806, 1557.708, 1833.702, 1823.738,
              1733.977, 1732.131, 1679.732, 1727.260, 1024.687,
              442.165,  1331.411, 1160.419],
    "S1GRD": [5.195, 5.890],
}
print("✅ Helpers and constants defined.")


---
# 📖 Section A — Theory  *(12 marks)*

> 📌 **Lab reference:** All theory answers can be found in **Lab 06, Cells 1–3** (GFM concepts, MAE, TerraMind architecture) and **Lab 06, Steps 2 & 6** (spectral analysis, evaluation metrics).


---
### Q1 — Type of Learning in MAE  *(2 marks)*

Masked Autoencoder (MAE) pre-training belongs to which type of learning? Select **one** and explain in one sentence why the others do not apply.

> 📌 *See Lab 06, Cell 2 — "How Prithvi Works — Vision Transformer + Masked Autoencoder"*

| Option | |
|--------|--|
| **A** | Supervised learning |
| **B** | Self-supervised learning |
| **C** | Reinforcement learning |
| **D** | Semi-supervised learning |


**Your answer:** *(replace this — state your choice and one-sentence justification)*


---
### Q2 — The Overall Idea of MAE  *(2 marks)*

In your own words, describe the overall idea of MAE pre-training covering: (a) what the model sees as input, (b) what it is asked to predict, (c) what useful knowledge it gains. Limit: 3–4 sentences.

> 📌 *See Lab 06, Cell 2 — the MAE diagram (INPUT → PATCHIFY → MASK → ENCODER → DECODER → LOSS)*


**Your answer:**

*(Write here)*


---
### Q3 — Popular Self-Supervised Learning Techniques  *(2 marks)*

Name **at least two** self-supervised learning techniques used in computer vision or Earth observation (other than MAE). One sentence description each.

> 📌 *See Lab 06, Cell 1 — SSL& GFM background reading links,


**Your answer:**

1. **Technique 1:** *(description)*

2. **Technique 2:** *(description)*


---
### Q4 — Advantages of Geospatial Foundation Models  *(3 marks)*

Name **three** potential advantages a GFM offers over training a task-specific model from scratch. One sentence each. *(1 mark each)*

> 📌 *See Lab 06, Cell 1 — "The Big Picture: From Task-Specific to Foundation Models"*


**Your answer:**

1. *(Advantage 1)*

2. *(Advantage 2)*

3. *(Advantage 3)*


---
### Q5 — IoU vs mIoU  *(1 mark)*

In one or two sentences, explain the difference between **IoU** and **mIoU**. When does the distinction matter most?

> 📌 *See Lab 06, Step 6 — "Evaluation: Segmentation Metrics Explained" with the IoU and mIoU formulas*


**Your answer:**

*(Write here)*


---
### Q6 — Advantage of Radar (Sentinel-1) in Flood Mapping  *(2 marks)*

Give **two distinct advantages** Sentinel-1 SAR data offers for flood mapping compared to optical imagery. *(1 mark each)*

> 📌 *See Lab 06, Step 1 — "Why two sensors?" and Step 2 — spectral analysis of SAR backscatter*


**Your answer:**

1. *(Advantage 1)*

2. *(Advantage 2)*


---
# 💻 Section B — Practical  *(13 marks)*

**Rules:**
- Fill in **only** the lines marked `### YOUR CODE HERE ###`
- Do **not** rename any variable — the auto-grader at the end depends on them
- Run cells in order — each task builds on the ones before
- Lab references are provided for every task


---
### P1 — Compute NDWI on Real Sen1Floods11 Chips  *(3 marks)*

The **Normalized Difference Water Index (NDWI)** highlights open water:

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

For Sentinel-2 L1C in our `s2` array: **Green = band index 2** (B3), **NIR = band index 7** (B8).

**Your tasks:**
1. Fill in the NDWI formula *(1 mark)*
2. Fill in the NDWI threshold for binary water detection *(1 mark)*
3. Compute the F1-score for your threshold-based prediction vs the hand label *(1 mark)*

> 📌 *See Lab 06, Step 1 (chip loading with `load_chip`) and Step 2 (NDWI separability plot, Cell 13)*


In [ ]:
# P1 — NDWI on real Sen1Floods11 data (3 marks)
# ─────────────────────────────────────────────────────────────────────────────
# We use the first test chip
chip_name = TEST_NAMES[0]
s2, s1, lbl = load_chip(chip_name)

green = s2[2]   # B3 Green
nir   = s2[7]   # B8 NIR

# ── Task 1 (1 mark): fill in the NDWI formula ────────────────────────────────
ndwi = ### YOUR CODE HERE ###
# Hint: (green - #) / ( # + nir + small_epsilon)
# Add 1e-8 to denominator to avoid division by zero


# ── Task 2 (1 mark): choose a threshold for binary water detection ────────────
# NDWI > threshold → predict flood (1), else predict non-flood (0)
# From the lab spectral analysis, what value separates water from land?
NDWI_THRESHOLD = ### YOUR CODE HERE ###
# Hint: look at the NDWI separability plot in Lab 06 Step 2 (Cell 13)
# The dashed yellow line in that plot marks the boundary


# Compute binary prediction from threshold
ndwi_pred = (ndwi > NDWI_THRESHOLD).astype(np.int16)

# ── Task 3 (1 mark): compute F1-score for flood class ─────────────────────────
valid = lbl >= 0   # exclude no-data pixels (label == -1)
tp = ((ndwi_pred == 1) & (lbl == 1) & valid).sum()
fp = ((ndwi_pred == 1) & (lbl == 0) & valid).sum()
fn = ((ndwi_pred == 0) & (lbl == 1) & valid).sum()

# Fill in the F1-score formula: F1 = 2*tp / (2*tp + fp + fn)
f1_ndwi =  ### YOUR CODE HERE ###
# Hint: use tp, fp, fn computed above; add 1e-8 to denominator


# ── Visualisation (do not edit) ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor("#0d1117")

event = chip_name.split("_")[0]
iou_ndwi = tp / (tp + fp + fn + 1e-8)

for ax, img, title, cm, vn, vx in [
    (axes[0], pct_stretch(np.stack([s2[3],s2[2],s2[1]],-1)), f"S2 RGB\n({event})", None, None, None),
    (axes[1], ndwi, f"NDWI (your formula)\nthreshold={NDWI_THRESHOLD}", "RdYlBu", -1, 1),
    (axes[2], np.where(valid, ndwi_pred, 0).astype(float), f"NDWI prediction\n(threshold={NDWI_THRESHOLD})", FLOOD_CMAP, 0, 1),
    (axes[3], np.where(valid, lbl, 0).astype(float), "Ground Truth", FLOOD_CMAP, 0, 1),
]:
    ax.set_facecolor("#161b22")
    ax.imshow(img, cmap=cm, vmin=vn, vmax=vx, interpolation="nearest")
    ax.set_title(title, color="white", fontsize=9, pad=6)
    ax.axis("off")

patches = [mpatches.Patch(color="#2d6a4f", label="Non-flood"),
           mpatches.Patch(color="#0077b6", label="Flood")]
fig.legend(handles=patches, loc="lower center", ncol=2, fontsize=9,
           facecolor="#21262d", labelcolor="white", bbox_to_anchor=(0.5,-0.02))
fig.suptitle(f"P1 — NDWI Analysis on Sen1Floods11 | IoU={iou_ndwi:.3f} | F1={f1_ndwi:.3f}",
             color="white", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("p1_ndwi.png", dpi=110, bbox_inches="tight", facecolor="#0d1117")
plt.show()

print(f"Chip      : {chip_name}")
print(f"TP={int(tp)}  FP={int(fp)}  FN={int(fn)}")
print(f"IoU (NDWI): {iou_ndwi:.3f}")
print(f"F1  (NDWI): {f1_ndwi:.3f}")


---
### P2 — Fine-tune TerraMind: Three Hyperparameter Settings  *(7 marks)*

You will run **three training experiments** with different hyperparameter combinations and compare results.
All model architecture and data loading code is provided — you only fill in the hyperparameters.

**Marks breakdown:**
- 1 mark per experiment for correct hyperparameter values (3 marks)
- 1 mark for the results table correctly filled (1 mark)
- 3 marks for written observations (1 per observation)

**Experiment settings to try:**

| Run | Epochs | Learning Rate | Loss Function |
|-----|--------|--------------|---------------|
| A   | 3      | 2e-4         | `ce`          |
| B   | 3      | 2e-5         | `dice`        |
| C   | 5      | 2e-5         | `focal`       |

> 📌 *See Lab 06, Steps 3–5: backbone loading (Cell 15), DataModule (Cell 17), SemanticSegmentationTask (Cell 20), Trainer (Cell 21), training (Cell 22)*


In [ ]:
# ── Shared DataModule (same for all three runs) ───────────────────────────────
# Full code from Lab 06 Cell 17 — do not edit
import albumentations as A
from albumentations.pytorch import ToTensorV2
from terratorch.datamodules import GenericMultiModalDataModule

DATA_ROOT = "sen1floods11_v1.1"

datamodule = GenericMultiModalDataModule(
    task="segmentation",
    num_classes=2,
    batch_size=4,
    num_workers=2,
    modalities=["S2L1C", "S1GRD"],
    rgb_indices={"S2L1C": [3, 2, 1]},
    train_data_root={"S2L1C": f"{DATA_ROOT}/data/S2L1CHand",
                     "S1GRD": f"{DATA_ROOT}/data/S1GRDHand"},
    val_data_root=  {"S2L1C": f"{DATA_ROOT}/data/S2L1CHand",
                     "S1GRD": f"{DATA_ROOT}/data/S1GRDHand"},
    test_data_root= {"S2L1C": f"{DATA_ROOT}/data/S2L1CHand",
                     "S1GRD": f"{DATA_ROOT}/data/S1GRDHand"},
    train_label_data_root=f"{DATA_ROOT}/data/LabelHand",
    val_label_data_root=  f"{DATA_ROOT}/data/LabelHand",
    test_label_data_root= f"{DATA_ROOT}/data/LabelHand",
    train_split=f"{DATA_ROOT}/splits/flood_train_data.txt",
    val_split=  f"{DATA_ROOT}/splits/flood_valid_data.txt",
    test_split= f"{DATA_ROOT}/splits/flood_test_data.txt",
    image_grep={"S2L1C": "*_S2Hand.tif", "S1GRD": "*_S1Hand.tif"},
    label_grep="*_LabelHand.tif",
    no_label_replace=-1,
    no_data_replace=0,
    means=MEANS,
    stds=STDS,
    train_transform=[A.D4(), ToTensorV2()],
    val_transform=[ToTensorV2()],
    test_transform=[ToTensorV2()],
)
datamodule.setup("fit")
datamodule.setup("test")
print(f"✅ DataModule ready | Train={len(datamodule.train_dataset)} Val={len(datamodule.val_dataset)} Test={len(datamodule.test_dataset)}")


In [ ]:
# ── Helper: build model + trainer for one experiment ──────────────────────────
# Full code adapted from Lab 06 Cells 20–21. Do not edit this cell.
import os
from terratorch.tasks import SemanticSegmentationTask
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, RichProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

def build_model(lr, loss_fn, run_name):
    m = SemanticSegmentationTask(
        model_factory="EncoderDecoderFactory",
        model_args={
            "backbone"             : BACKBONE,
            "backbone_pretrained"  : True,
            "backbone_modalities"  : ["S2L1C", "S1GRD"],
            "backbone_merge_method": "mean",
            "necks": [
                {"name": "SelectIndices",               "indices": [2, 5, 8, 11]},
                {"name": "ReshapeTokensToImage",         "remove_cls_token": False},
                {"name": "LearnedInterpolateToPyramidal"},
            ],
            "decoder"         : "UNetDecoder",
            "decoder_channels": [256, 128, 64, 32],
            "head_dropout"    : 0.1,
            "num_classes"     : 2,
        },
        loss=loss_fn,
        ignore_index=-1,
        freeze_backbone=True,
        freeze_decoder=False,
        optimizer="AdamW",
        lr=lr,
        class_names=["Other", "Flood"],
        plot_on_val=False,
    )
    return m

def build_trainer(max_epochs, run_name):
    ckpt_dir = f"output/{run_name}/checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_cb = ModelCheckpoint(
        dirpath=ckpt_dir,
        filename="best-mIoU",
        monitor="val/mIoU",
        mode="max",
        save_top_k=1,
    )
    t = pl.Trainer(
        accelerator="auto",
        devices="auto",
        precision="16-mixed",
        max_epochs=max_epochs,
        log_every_n_steps=5,
        callbacks=[ckpt_cb, LearningRateMonitor("epoch"), RichProgressBar()],
        logger=TensorBoardLogger(f"output/{run_name}/", name="tb"),
        default_root_dir=f"output/{run_name}/",
        enable_progress_bar=True,
    )
    return t, ckpt_cb

print("✅ Helper functions defined.")


In [ ]:
# ── Run A: Cross-Entropy loss, high LR ───────────────────────────────────────
print("=" * 55)
print("RUN A  |  epochs=3  |  lr=2e-4  |  loss=ce")
print("=" * 55)

# ── Fill in hyperparameters for Run A (1 mark) ───────────────────────────────
EPOCHS_A = ### YOUR CODE HERE ###   # number of training epochs
LR_A     = ### YOUR CODE HERE ###   # learning rate
LOSS_A   = ### YOUR CODE HERE ###   # loss function name (string)


model_A   = build_model(lr=LR_A, loss_fn=LOSS_A, run_name="run_A")
trainer_A, ckpt_A = build_trainer(max_epochs=EPOCHS_A, run_name="run_A")
trainer_A.fit(model_A, datamodule=datamodule)

results_A = trainer_A.test(model_A, datamodule=datamodule,
                            ckpt_path=ckpt_A.best_model_path)
mIoU_A    = results_A[0].get("test/mIoU", results_A[0].get("test/Multiclass_Jaccard_Index", 0.0))
print(f"\nRun A — test mIoU: {mIoU_A:.4f}")


In [ ]:
# ── Run B: Dice loss, lower LR ────────────────────────────────────────────────
print("=" * 55)
print("RUN B  |  epochs=3  |  lr=2e-5  |  loss=dice")
print("=" * 55)

# ── Fill in hyperparameters for Run B (1 mark) ───────────────────────────────
EPOCHS_B = ### YOUR CODE HERE ###
LR_B     = ### YOUR CODE HERE ###
LOSS_B   = ### YOUR CODE HERE ###


model_B   = build_model(lr=LR_B, loss_fn=LOSS_B, run_name="run_B")
trainer_B, ckpt_B = build_trainer(max_epochs=EPOCHS_B, run_name="run_B")
trainer_B.fit(model_B, datamodule=datamodule)

results_B = trainer_B.test(model_B, datamodule=datamodule,
                            ckpt_path=ckpt_B.best_model_path)
mIoU_B    = results_B[0].get("test/mIoU", results_B[0].get("test/Multiclass_Jaccard_Index", 0.0))
print(f"\nRun B — test mIoU: {mIoU_B:.4f}")


In [ ]:
# ── Run C: Focal loss, more epochs ────────────────────────────────────────────
print("=" * 55)
print("RUN C  |  epochs=5  |  lr=2e-5  |  loss=focal")
print("=" * 55)

# ── Fill in hyperparameters for Run C (1 mark) ───────────────────────────────
EPOCHS_C = ### YOUR CODE HERE ###
LR_C     = ### YOUR CODE HERE ###
LOSS_C   = ### YOUR CODE HERE ###


model_C   = build_model(lr=LR_C, loss_fn=LOSS_C, run_name="run_C")
trainer_C, ckpt_C = build_trainer(max_epochs=EPOCHS_C, run_name="run_C")
trainer_C.fit(model_C, datamodule=datamodule)

results_C = trainer_C.test(model_C, datamodule=datamodule,
                            ckpt_path=ckpt_C.best_model_path)
mIoU_C    = results_C[0].get("test/mIoU", results_C[0].get("test/Multiclass_Jaccard_Index", 0.0))
print(f"\nRun C — test mIoU: {mIoU_C:.4f}")


In [ ]:
# ── Results table — auto-generated (do not edit) ─────────────────────────────
print("\n" + "=" * 65)
print("HYPERPARAMETER EXPERIMENT RESULTS")
print("=" * 65)
print(f"{'Run':<6} {'Epochs':>7} {'LR':>8} {'Loss':>8} {'Test mIoU':>10}")
print("-" * 65)
print(f"{'A':<6} {EPOCHS_A:>7} {LR_A:>8} {LOSS_A:>8} {mIoU_A:>10.4f}")
print(f"{'B':<6} {EPOCHS_B:>7} {LR_B:>8} {LOSS_B:>8} {mIoU_B:>10.4f}")
print(f"{'C':<6} {EPOCHS_C:>7} {LR_C:>8} {LOSS_C:>8} {mIoU_C:>10.4f}")
print("=" * 65)

best_run = max([("A", mIoU_A), ("B", mIoU_B), ("C", mIoU_C)], key=lambda x: x[1])
print(f"\nBest run: {best_run[0]}  (mIoU = {best_run[1]:.4f})")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")
runs   = ["Run A\n(ce, lr=2e-4, 3ep)", "Run B\n(dice, lr=2e-5, 3ep)", "Run C\n(focal, lr=2e-5, 5ep)"]
mious  = [mIoU_A, mIoU_B, mIoU_C]
colors = ["#ef5350", "#42a5f5", "#66bb6a"]
bars   = ax.bar(runs, mious, color=colors, edgecolor="white", linewidth=0.7, alpha=0.85)
for bar, v in zip(bars, mious):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{v:.4f}", ha="center", va="bottom", color="white", fontsize=10, fontweight="bold")
ax.set_ylabel("Test mIoU", color="#8b949e")
ax.set_title("P2 — Hyperparameter Comparison: TerraMind on Sen1Floods11",
             color="white", fontsize=11, fontweight="bold")
ax.set_ylim(0, min(1.0, max(mious) + 0.15))
ax.tick_params(colors="#8b949e")
ax.grid(True, axis="y", color="#21262d", lw=0.5)
for s in ax.spines.values(): s.set_color("#30363d")
plt.tight_layout()
plt.savefig("p2_comparison.png", dpi=120, bbox_inches="tight", facecolor="#0d1117")
plt.show()


---
#### P2 — Written Observations  *(3 marks, 1 per observation)*

Based on your three runs, write **three observations** in the cell below.
Each observation should relate a specific hyperparameter change to the result you saw.

> 📌 *See Lab 06, Step 5 — "Fine-tuning recipe" for guidance on learning rate, loss, and epochs*

**Examples of good observations** (do not copy these — write your own based on your actual numbers):
- *"Run B outperformed Run A despite fewer parameters being changed, suggesting that..."*
- *"Increasing epochs from 3 to 5 in Run C resulted in... because..."*
- *"The Dice loss in Run B vs Cross-Entropy in Run A shows... which makes sense for imbalanced flood data because..."*


**Observations:**

1. *(Observation 1 — relate a hyperparameter change to the result)*

2. *(Observation 2 — relate a hyperparameter change to the result)*

3. *(Observation 3 — relate a hyperparameter change to the result)*


---
### P3 — Visualise Best Model Predictions & Compute IoU  *(3 marks)*

Using the best-performing model from P2, visualise predictions on test chips and compute the per-chip IoU.

**Your tasks:**
1. Load the best model checkpoint *(1 mark)*
2. Fill in the TP / FP / FN calculation to compute IoU *(1 mark)*
3. Compare your best TerraMind IoU to the NDWI F1 from P1 and write one sentence *(1 mark)*

> 📌 *See Lab 06, Steps 6 — Cells 25, 26, and 27 for inference code and IoU calculation*


In [ ]:
# ── Task 1 (1 mark): load the best checkpoint ────────────────────────────────
# Identify which run had the best mIoU and load that checkpoint

best_run_name = ### YOUR CODE HERE ###
# Fill in "run_A", "run_B", or "run_C" — whichever had the highest test mIoU

best_ckpt_path = f"output/{best_run_name}/checkpoints/best-mIoU.ckpt"

# Which model object corresponds to that run? (do not edit the logic, just the variable)
best_model_obj = {"run_A": model_A, "run_B": model_B, "run_C": model_C}[best_run_name]

from terratorch.tasks import SemanticSegmentationTask
inf_model = SemanticSegmentationTask.load_from_checkpoint(
    best_ckpt_path,
    model_factory=best_model_obj.hparams["model_factory"],
    model_args=best_model_obj.hparams["model_args"],
)
inf_model.eval().to(DEVICE)
print(f"✅ Loaded: {best_ckpt_path}")
print(f"   Run   : {best_run_name}")


In [ ]:
# ── Task 2 (1 mark): compute per-chip IoU on test set ───────────────────────
import torch, torch.nn.functional as F

datamodule.setup("test")
test_loader = datamodule.test_dataloader()

all_ious = []
all_samples = []

with torch.no_grad():
    for batch in test_loader:
        s2b  = batch["image"]["S2L1C"].to(DEVICE)
        s1b  = batch["image"]["S1GRD"].to(DEVICE)
        lblb = batch["mask"]

        out    = inf_model.model({"S2L1C": s2b, "S1GRD": s1b})
        logits = out.output if hasattr(out,"output") else (out[0] if isinstance(out,(list,tuple)) else out)
        preds  = logits.argmax(1).cpu()

        for i in range(s2b.shape[0]):
            gt_i   = lblb[i].numpy()
            pred_i = preds[i].numpy()
            valid  = gt_i >= 0

            # ── Fill in TP, FP, FN (1 mark) ──────────────────────────────────
            tp = ### YOUR CODE HERE ###
            # pixels predicted flood AND actually flood AND valid
            fp = ### YOUR CODE HERE ###
            # pixels predicted flood AND actually NOT flood AND valid
            fn = ### YOUR CODE HERE ### e.g., ((pred_i == 0) & (gt_i == 1) & valid).sum()
            # pixels NOT predicted flood AND actually flood AND valid

            iou_i = tp / (tp + fp + fn + 1e-8)
            all_ious.append(float(iou_i))
            all_samples.append({
                "s2": s2b[i].cpu(), "s1": s1b[i].cpu(),
                "gt": gt_i, "pred": pred_i, "iou": float(iou_i),
            })

mean_test_iou = np.mean(all_ious)
print(f"Test chips: {len(all_ious)}")
print(f"Mean IoU (flood): {mean_test_iou:.4f}")
print(f"Min IoU: {min(all_ious):.4f} | Max IoU: {max(all_ious):.4f}")


In [ ]:
# ── Visualise best and worst predictions (do not edit) ───────────────────────
all_samples_sorted = sorted(all_samples, key=lambda x: x["iou"], reverse=True)
n_show = 4   # top 2 + bottom 2
show_samples = all_samples_sorted[:2] + all_samples_sorted[-2:]
labels_show  = ["Best #1", "Best #2", "Worst #2", "Worst #1"]

def denorm_s2(t):
    m = np.array(MEANS["S2L1C"])[:,None,None]
    s = np.array(STDS["S2L1C"])[:,None,None]
    return t.numpy() * s + m

fig, axes = plt.subplots(n_show, 5, figsize=(22, 4.5*n_show))
fig.patch.set_facecolor("#0d1117")
col_titles = ["S2 RGB", "S1 VV (SAR)", "Ground Truth", "Prediction", "Error Map"]

for row_i, (samp, rank_label) in enumerate(zip(show_samples, labels_show)):
    s2_raw = denorm_s2(samp["s2"])
    gt     = samp["gt"]
    pred   = samp["pred"]
    valid  = gt >= 0
    s1n    = samp["s1"].numpy()

    rgb  = pct_stretch(np.stack([s2_raw[3],s2_raw[2],s2_raw[1]],-1))
    vv   = pct_stretch(s1n[0])
    err  = np.full((*gt.shape,3), 0.1)
    err[(gt==1)&(pred==1)&valid] = [0.0, 0.8, 0.2]
    err[(gt==0)&(pred==0)&valid] = [0.1, 0.1, 0.1]
    err[(gt==0)&(pred==1)&valid] = [0.9, 0.1, 0.1]
    err[(gt==1)&(pred==0)&valid] = [0.9, 0.8, 0.0]

    for ci, (img, cm, vn, vx) in enumerate([
        (rgb,                                     None,       None, None),
        (vv,                                      "gray",     None, None),
        (np.where(valid,gt,0).astype(float),      FLOOD_CMAP,   0,    1),
        (np.where(valid,pred,0).astype(float),    FLOOD_CMAP,   0,    1),
        (err,                                     None,       None, None),
    ]):
        ax = axes[row_i, ci]
        ax.set_facecolor("#161b22")
        ax.imshow(img, cmap=cm, vmin=vn, vmax=vx, interpolation="nearest")
        ax.axis("off")
        if row_i == 0:
            ax.set_title(col_titles[ci], color="#80cbc4", fontsize=10, fontweight="bold", pad=8)

    axes[row_i, 0].text(5, 20, f"{rank_label}\nIoU={samp['iou']:.3f}",
                        color="white", fontsize=9, fontweight="bold",
                        bbox=dict(fc="black", alpha=0.6, pad=2))

patches_legend = [
    mpatches.Patch(color=(0,0.8,0.2),   label="TP: Correct flood"),
    mpatches.Patch(color=(0.1,0.1,0.1), label="TN: Correct land"),
    mpatches.Patch(color=(0.9,0.1,0.1), label="FP: False alarm"),
    mpatches.Patch(color=(0.9,0.8,0),   label="FN: Missed flood"),
]
fig.legend(handles=patches_legend, loc="lower center", ncol=4, fontsize=9,
           facecolor="#21262d", labelcolor="white", bbox_to_anchor=(0.5,-0.02))
fig.suptitle(f"P3 — TerraMind Predictions (Run {best_run_name}) | Mean IoU={mean_test_iou:.4f}",
             color="white", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("p3_predictions.png", dpi=110, bbox_inches="tight", facecolor="#0d1117")
plt.show()


#### P3 — Comparison: TerraMind vs NDWI  *(1 mark)*

In the cell below, write **one sentence** comparing the TerraMind mean IoU (from P3) with the NDWI F1-score (from P1).
Comment on what this difference tells you about learned vs rule-based methods.

> 📌 *See Lab 06, Step 2 — spectral analysis discussion ("NDWI > 0: classical water rule — but imperfect")*


**Your comparison (1 sentence):**

*(Write here — e.g. "TerraMind achieved ... vs NDWI F1 of ..., which shows that ... because ...")*


---
## 📤 Submission

1. Ensure **all cells are executed** (no empty outputs).
2. `File → Save` then `File → Download → Download .ipynb`.
3. Rename to `Lab06_Assessment_<YourNetID>.ipynb` and submit on Canvas.

| Question | Topic | Marks |
|----------|-------|-------|
| Q1 | MAE learning type | 2 |
| Q2 | MAE overall idea | 2 |
| Q3 | SSL techniques | 2 |
| Q4 | GFM advantages | 3 |
| Q5 | IoU vs mIoU | 1 |
| Q6 | SAR advantages | 2 |
| P1 | NDWI on real chips + F1 | 3 |
| P2 | Three training runs | 4 |
| P2 | Written observations | 3 |
| P3 | Load best model + IoU + comparison | 3 |
| **Total** | | **25** |
